In [2]:
!pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable


In [32]:
import os
import psycopg2
sql_dir = r"C:\Users\chapman\Downloads\vkr_github\sql"  


print("Содержимое папки sql:", os.listdir(sql_dir))

conn = psycopg2.connect(
    host="localhost",
    port="5432",
    dbname="postgres",
    user="postgres",
    password="postgres"
)
conn.autocommit = True  
scripts = [
    "01_create_schemas.sql",
    "02_create_staging_tables.sql",
    "03_create_dw_tables.sql"
]

with conn.cursor() as cur:
    for script in scripts:
        filepath = os.path.join(sql_dir, script)
        with open(filepath, "r", encoding="utf-8") as f:
            sql = f.read()
        cur.execute(sql)
        print("Готово.\n")

conn.close()

Содержимое папки sql: ['01_create_schemas.sql', '02_create_staging_tables.sql', '03_create_dw_tables.sql', '04_load_csv_to_staging.sql', '05_etl.sql', '06_analytics_views.sql', '07_analytics_queries.sql']
Готово.

Готово.

Готово.



In [33]:
import os
import psycopg2

data_dir = r"C:\Users\chapman\Downloads\vkr_github\data"   


print("Содержимое папки data:", os.listdir(data_dir))


conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "postgres"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "postgres")
)

with conn:
    with conn.cursor() as cur:
        files = [
            ("stg.products",       os.path.join(data_dir, "products.csv")),
            ("stg.equipment",      os.path.join(data_dir, "equipment.csv")),
            ("stg.employees",      os.path.join(data_dir, "employees.csv")),
            ("stg.orders",         os.path.join(data_dir, "orders.csv")),
            ("stg.operations",     os.path.join(data_dir, "operations.csv")),
            ("stg.defects",        os.path.join(data_dir, "defects.csv")),
            ("stg.downtime",       os.path.join(data_dir, "downtime.csv")),
        ]
        for table, path in files:  
            with open(path, "r", encoding="utf-8-sig") as f:
                cur.copy_expert(f"COPY {table} FROM STDIN WITH CSV HEADER", f)

print("Загрузка CSV в staging завершена")

Содержимое папки data: ['defects.csv', 'downtime.csv', 'employees.csv', 'equipment.csv', 'operations.csv', 'orders.csv', 'products.csv']
Загрузка CSV в staging завершена
